# LLM Subqueries por QueryDocumentId + BM25S

Este notebook:
- carrega proposições geradas por LLM a partir de fragmentos de query;
- cria subqueries com base no campo `proposition`;
- indexa o corpus com BM25S usando tokens já processados;
- soma os scores dos documentos recuperados por cada subquery;
- retorna os top 100 documentos por queryDocumentId;
- calcula métricas de precisão dos noticedId retornados.

In [ ]:
%pip install "bm25s[full]"

In [ ]:
import json
import os
from collections import defaultdict

import bm25s

In [ ]:
BASE_PATH = 'data'
CACHE_DIR = os.path.join(BASE_PATH, 'bm25/bm25-all-paragraphs-returns-document-id/cache')

PROPOSITIONS_FILE = os.path.join(BASE_PATH, 'query_propositions_with_llm.json')
LEGACY_PROPOSITIONS_FILE = os.path.join(BASE_PATH, 'query_proposotions_with_llm.json')
CORPUS_CACHE_FILE = os.path.join(CACHE_DIR, 'corpus_paragraphs_all.json')
LABELS_FILE = os.path.join(BASE_PATH, 'task1_train_labels_2025.json')

BM25_TOP_N_DOCS = 100
BM25_METHOD = 'bm25l'
BM25_K1 = 3.5
BM25_B = 1

# Quantidade de documentos por subquery antes da agregacao.
# Um valor maior ajuda a aumentar cobertura antes do top 100 final.
PER_SUBQUERY_K = 300

RESULTS_OUTPUT_FILE = os.path.join(BASE_PATH, 'bm25/llm-query/retrieval_top100_by_query.json')
EVALUATION_OUTPUT_FILE = os.path.join(BASE_PATH, 'bm25/llm-query/evaluation_summary.json')

print(f'PROPOSITIONS_FILE: {PROPOSITIONS_FILE}')
print(f'LEGACY_PROPOSITIONS_FILE: {LEGACY_PROPOSITIONS_FILE}')
print(f'CORPUS_CACHE_FILE: {CORPUS_CACHE_FILE}')
print(f'LABELS_FILE: {LABELS_FILE}')
print(f'BM25_TOP_N_DOCS: {BM25_TOP_N_DOCS}')
print(f'BM25_METHOD: {BM25_METHOD}')
print(f'BM25_K1: {BM25_K1}')
print(f'BM25_B: {BM25_B}')
print(f'PER_SUBQUERY_K: {PER_SUBQUERY_K}')
print(f'RESULTS_OUTPUT_FILE: {RESULTS_OUTPUT_FILE}')
print(f'EVALUATION_OUTPUT_FILE: {EVALUATION_OUTPUT_FILE}')

In [ ]:
def input_propositions_file(path: str) -> str:
    if os.path.exists(path):
        return path
    raise FileNotFoundError(
        f'Arquivo de proposicoes nao encontrado em: {path}'
    )


def normalize_case_id(case_id) -> str:
    if case_id is None:
        return ''

    case_id = str(case_id).strip()
    if not case_id:
        return ''

    if case_id.lower().endswith('.txt'):
        case_id = case_id[:-4]

    return case_id


def extract_query_doc_id(item: dict) -> str:
    candidates = [
        item.get('doc_id'),
    ]

    for candidate in candidates:
        normalized = normalize_case_id(candidate)
        if normalized:
            return normalized
    return ''


def load_inputs(propositions_file: str, corpus_cache_file: str):
    with open(propositions_file, 'r', encoding='utf-8') as f:
        propositions_payload = json.load(f)

    with open(corpus_cache_file, 'r', encoding='utf-8') as f:
        corpus_items = json.load(f)

    items = propositions_payload.get('items', [])
    print(f'Proposicoes carregadas: {len(items)}')
    print(f'Corpus paragrafos carregados: {len(corpus_items)}')
    return items, corpus_items


def build_subqueries_by_doc_id(proposition_items):
    subqueries_by_doc_id = defaultdict(list)

    for item in proposition_items:
        doc_id = extract_query_doc_id(item)
        proposition = str(item.get('proposition', '')).strip()

        if not doc_id or not proposition:
            continue

        subqueries_by_doc_id[doc_id].append({
            'paragraph_id': item.get('paragraph_id'),
            'subquery': proposition,
            'cleaned_text': item.get('cleaned_text', ''),
        })

    print(f'Query documents com subqueries: {len(subqueries_by_doc_id)}')
    return dict(subqueries_by_doc_id)


def build_corpus_bm25_index(corpus_items, method='bm25l', k1=1.2, b=0.4):
    doc_tokens_map = defaultdict(list)

    for item in corpus_items:
        doc_id = normalize_case_id(item['doc_id'])
        doc_tokens_map[doc_id].extend(item.get('processed_tokens', []))

    corpus_doc_ids = list(doc_tokens_map.keys())
    corpus_tokens = [doc_tokens_map[doc_id] for doc_id in corpus_doc_ids]

    retriever = bm25s.BM25(method=method, k1=k1, b=b)
    retriever.index(corpus_tokens)

    print(f'Documentos indexados no BM25: {len(corpus_doc_ids)}')
    return retriever, corpus_doc_ids


def evaluate_against_labels(retrieval_results, labels_dict):
    per_query_metrics = {}
    total_relevant = 0
    total_retrieved_relevant = 0
    total_retrieved = 0

    for query_case_id_raw, noticed_ids_raw in labels_dict.items():
        query_case_id = normalize_case_id(query_case_id_raw)
        relevant_ids = {normalize_case_id(x) for x in noticed_ids_raw}
        relevant_ids.discard('')

        retrieved_docs = retrieval_results.get(query_case_id, {}).get('top_docs', [])
        retrieved_ids = {normalize_case_id(item['doc_id']) for item in retrieved_docs}
        retrieved_ids.discard('')

        relevant_count = len(relevant_ids)
        retrieved_count = len(retrieved_ids)
        retrieved_relevant_count = len(relevant_ids.intersection(retrieved_ids))

        total_relevant += relevant_count
        total_retrieved_relevant += retrieved_relevant_count
        total_retrieved += retrieved_count

        precision_at_k = (retrieved_relevant_count / retrieved_count) if retrieved_count > 0 else 0.0
        recall = (retrieved_relevant_count / relevant_count) if relevant_count > 0 else 0.0

        per_query_metrics[query_case_id] = {
            'precision_at_k': precision_at_k,
            'recall': recall,
            'relevant_total': relevant_count,
            'retrieved_total': retrieved_count,
            'relevant_retrieved': retrieved_relevant_count,
            'relevant_missing': relevant_count - retrieved_relevant_count,
        }

    summary = {
        'overall_precision_at_k': (total_retrieved_relevant / total_retrieved) if total_retrieved > 0 else 0.0,
        'overall_recall': (total_retrieved_relevant / total_relevant) if total_relevant > 0 else 0.0,
        'total_relevant_cases': total_relevant,
        'total_retrieved_relevant_cases': total_retrieved_relevant,
        'total_missing_relevant_cases': total_relevant - total_retrieved_relevant,
        'total_retrieved_cases': total_retrieved,
        'queries_evaluated': len(labels_dict),
    }

    return per_query_metrics, summary

In [ ]:
def retrieve_top_docs_from_subqueries(
    subqueries_by_doc_id,
    retriever,
    corpus_doc_ids,
    top_n_docs=100,
    per_subquery_k=300,
):
    all_results = {}

    for query_doc_id, subqueries in subqueries_by_doc_id.items():
        score_by_doc = defaultdict(float)
        hits_by_doc = defaultdict(int)
        details = []

        for sq in subqueries:
            query_text = sq['subquery']
            query_tokens = bm25s.tokenize(query_text, stopwords='en')
            results, scores = retriever.retrieve(list(query_tokens), k=per_subquery_k)

            subquery_top_docs = []
            for i in range(results.shape[1]):
                idx = int(results[0, i])
                doc_id = normalize_case_id(corpus_doc_ids[idx])
                score = float(scores[0, i])

                # Soma dos scores por documento ao longo de todas as subqueries.
                score_by_doc[doc_id] += score
                hits_by_doc[doc_id] += 1

                if i < top_n_docs:
                    subquery_top_docs.append({'doc_id': doc_id, 'score': score})

            details.append({
                'paragraph_id': sq.get('paragraph_id'),
                'subquery': query_text,
                'top_docs': subquery_top_docs,
            })

        ranked_docs = sorted(score_by_doc.items(), key=lambda x: x[1], reverse=True)
        top_docs = ranked_docs[:top_n_docs]

        all_results[query_doc_id] = {
            'top_docs': [
                {
                    'doc_id': doc_id,
                    'score': float(score),
                    'matched_subqueries': int(hits_by_doc[doc_id]),
                }
                for doc_id, score in top_docs
            ],
            'subqueries': details,
        }

    return all_results

In [ ]:
resolved_propositions_file = input_propositions_file(
    PROPOSITIONS_FILE,
)

print(f'Arquivo de proposicoes em uso: {resolved_propositions_file}')
proposition_items, corpus_items = load_inputs(resolved_propositions_file, CORPUS_CACHE_FILE)

subqueries_by_doc_id = build_subqueries_by_doc_id(proposition_items)

retriever, corpus_doc_ids = build_corpus_bm25_index(
    corpus_items,
    method=BM25_METHOD,
    k1=BM25_K1,
    b=BM25_B,
)

retrieval_results = retrieve_top_docs_from_subqueries(
    subqueries_by_doc_id=subqueries_by_doc_id,
    retriever=retriever,
    corpus_doc_ids=corpus_doc_ids,
    top_n_docs=BM25_TOP_N_DOCS,
    per_subquery_k=PER_SUBQUERY_K,
)

os.makedirs(os.path.dirname(RESULTS_OUTPUT_FILE), exist_ok=True)
with open(RESULTS_OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(retrieval_results, f, ensure_ascii=True, indent=2)

with open(LABELS_FILE, 'r', encoding='utf-8') as f:
    labels_dict = json.load(f)

per_query_metrics, evaluation_summary = evaluate_against_labels(
    retrieval_results=retrieval_results,
    labels_dict=labels_dict,
)

with open(EVALUATION_OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(
        {
            'summary': evaluation_summary,
            'per_query': per_query_metrics,
        },
        f,
        ensure_ascii=True,
        indent=2,
    )

print(f'Queries executadas: {len(retrieval_results)}')
print(f'Resultados salvos em: {RESULTS_OUTPUT_FILE}')
print(f'Avaliacao salva em: {EVALUATION_OUTPUT_FILE}')
print('')
print('Resumo de precision/recall noticedId:')
print(f"Precision@{BM25_TOP_N_DOCS} geral: {evaluation_summary['overall_precision_at_k']:.2%}")
print(f"Recall geral: {evaluation_summary['overall_recall']:.2%}")
print(f"Casos notaveis totais: {evaluation_summary['total_relevant_cases']}")
print(f"Casos notaveis recuperados: {evaluation_summary['total_retrieved_relevant_cases']}")
print(f"Casos notaveis faltantes: {evaluation_summary['total_missing_relevant_cases']}")

In [ ]:
sample_query_doc_id = next(iter(retrieval_results.keys()))
sample = retrieval_results[sample_query_doc_id]

print(f'QueryDocumentId de exemplo: {sample_query_doc_id}')
print(f'Quantidade de subqueries: {len(sample["subqueries"])}')

print('\n' + '=' * 80)
print('TOP 100 DOCUMENTOS COM MAIOR SCORE SOMADO')
print('=' * 80)
for rank, item in enumerate(sample['top_docs'], start=1):
    print(
        f"[Rank {rank}] doc_id={item['doc_id']} | score_somado={item['score']:.4f} | matched_subqueries={item['matched_subqueries']}"
    )

if sample_query_doc_id in per_query_metrics:
    query_metrics = per_query_metrics[sample_query_doc_id]
    print('\n' + '=' * 80)
    print('METRICAS DA QUERY DE EXEMPLO')
    print('=' * 80)
    print(f"Precision@{BM25_TOP_N_DOCS}: {query_metrics['precision_at_k']:.2%}")
    print(f"Recall: {query_metrics['recall']:.2%}")
    print(f"Relevant total: {query_metrics['relevant_total']}")
    print(f"Relevant retrieved: {query_metrics['relevant_retrieved']}")